# Conditional GASP Test Notebook

This notebook tests and evaluates the Conditional GASP model against standard GASP
across different tissue types with varying T2/T1 ratios.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

# Check for PyTorch
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not installed. Install with: pip install torch")
    raise

## 1. Test Basic Functionality

In [ ]:
from gasp.simulation import SSFPParams
from gasp.responses import gaussian
from gasp.ml.conditional_gasp import ConditionalGASP, design_matrix_torch
from gasp.ml.data_generator import generate_training_batch, generate_signal_simple
from gasp.ml.losses import conditional_gasp_loss

def create_acquisition_params(n_pcs: int = 8, n_TRs: int = 3):
    """Create standard acquisition parameters."""
    TRs = [5e-3, 10e-3, 15e-3][:n_TRs]
    pcs = np.linspace(0, 2 * np.pi, n_pcs, endpoint=False)
    
    length = n_pcs * n_TRs
    alpha_list = [np.deg2rad(60)] * length
    TR_list = []
    pc_list = []
    for tr in TRs:
        for pc in pcs:
            TR_list.append(tr)
            pc_list.append(pc)
    
    return SSFPParams(length, alpha_list, TR_list, pc_list)

# Setup
params = create_acquisition_params(n_pcs=8, n_TRs=3)
n_acquisitions = params.length
width = 128
batch_size = 8

print(f"Number of acquisitions: {n_acquisitions}")
print(f"Spectral width: {width}")

In [ ]:
# Create model
model = ConditionalGASP(
    n_acquisitions=n_acquisitions,
    latent_dim=16,
    method="affine"
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {n_params:,} parameters")
print(f"Number of coefficients: {model.n_coeffs}")

In [ ]:
# Generate test batch
signals, tissue_params = generate_training_batch(
    batch_size, params, width=width,
    t1_range=(0.1, 4.0),
    t2_t1_ratio_range=(0.01, 0.5)
)

print(f"Generated batch:")
print(f"  Signals shape: {signals.shape}")
print(f"  Tissue params shape: {tissue_params.shape}")
print(f"\nSample tissue parameters (T1, T2, T2/T1):")
for i in range(min(3, batch_size)):
    print(f"  Sample {i}: T1={tissue_params[i,0]:.3f}s, T2={tissue_params[i,1]:.4f}s, ratio={tissue_params[i,2]:.4f}")

In [ ]:
# Forward pass
signals_flat = torch.from_numpy(signals.reshape(batch_size * width, -1))
A, z = model(signals_flat, return_latent=True)

print(f"Forward pass results:")
print(f"  Coefficients A shape: {A.shape}")
print(f"  Latent z shape: {z.shape}")

# Apply GASP
Phi = design_matrix_torch(signals_flat, model.method)
output = (Phi * A).sum(dim=-1)
print(f"  Output shape: {output.shape}")

## 2. Visualize bSSFP Signals for Different Tissues

In [ ]:
# Test tissues with different T2/T1 ratios
test_tissues = [
    ('Water', 4.0, 2.0, 'blue'),
    ('Gray Matter', 0.9, 0.1, 'green'),
    ('White Matter', 0.6, 0.08, 'orange'),
    ('Muscle', 0.9, 0.05, 'red'),
    ('Tendon', 0.4, 0.005, 'purple'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot magnitude
ax1 = axes[0]
for name, t1, t2, color in test_tissues:
    signal = generate_signal_simple(t1, t2, params, width=width)
    # Average over acquisitions for visualization
    mag = np.abs(signal).mean(axis=1)
    ax1.plot(mag, label=f"{name} (T2/T1={t2/t1:.3f})", color=color)

ax1.set_xlabel('Frequency Index')
ax1.set_ylabel('Magnitude')
ax1.set_title('bSSFP Signal Magnitude (averaged over acquisitions)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot single acquisition
ax2 = axes[1]
for name, t1, t2, color in test_tissues:
    signal = generate_signal_simple(t1, t2, params, width=width)
    # First acquisition
    ax2.plot(np.abs(signal[:, 0]), label=f"{name}", color=color, alpha=0.7)

ax2.set_xlabel('Frequency Index')
ax2.set_ylabel('Magnitude')
ax2.set_title('bSSFP Signal (first acquisition)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Train Conditional GASP Model

In [ ]:
from gasp.ml.trainer import train_conditional_gasp

# Create target profile
target_profile = gaussian(width, bw=0.25, shift=0)

plt.figure(figsize=(10, 4))
plt.plot(target_profile)
plt.xlabel('Frequency Index')
plt.ylabel('Amplitude')
plt.title('Target Spectral Profile (Gaussian)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Train the model
print("Training Conditional GASP model...")
print("This will take a few minutes.\n")

model, history = train_conditional_gasp(
    params=params,
    target_profile=target_profile,
    n_epochs=50,
    batch_size=32,
    n_batches_per_epoch=50,
    learning_rate=1e-3,
    width=width,
    latent_dim=16,
    method="affine",
    t1_range=(0.1, 4.0),
    t2_t1_ratio_range=(0.01, 0.5),
    noise_sigma=0.005,
    lambda_l2=1e-3,
    lambda_smooth=1e-4,
    verbose=True,
)

print("\nTraining complete!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(history.train_loss, label='Train Loss')
ax1.plot(history.val_loss, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training History')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(history.profile_loss, label='Profile Loss')
ax2.plot(history.l2_loss, label='L2 Loss')
ax2.plot(history.smooth_loss, label='Smooth Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss Components')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Compare with Standard GASP

In [ ]:
from gasp.gasp import train_gasp, run_gasp

# Train standard GASP on gray matter (reference tissue)
ref_t1, ref_t2 = 0.9, 0.1
ref_signal = generate_signal_simple(ref_t1, ref_t2, params, width=width)
ref_signal_2d = ref_signal.reshape(width, 1, -1)

_, standard_coeffs = train_gasp(
    ref_signal_2d, target_profile,
    method="affine", useL2=True, lam=1e-2
)

print(f"Standard GASP trained on Gray Matter (T1={ref_t1}s, T2={ref_t2}s)")
print(f"Coefficients shape: {standard_coeffs.shape}")

In [ ]:
# Evaluate both methods on all test tissues
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

results = {
    'tissue': [],
    't1': [],
    't2': [],
    'ratio': [],
    'standard_mse': [],
    'conditional_mse': [],
    'standard_output': [],
    'conditional_output': [],
}

test_tissues_eval = [
    ('Water', 4.0, 2.0),
    ('Gray Matter', 0.9, 0.1),
    ('White Matter', 0.6, 0.08),
    ('Muscle', 0.9, 0.05),
    ('Fat', 0.25, 0.07),
    ('Liver', 0.5, 0.04),
    ('Tendon', 0.4, 0.005),
]

print("Evaluating on test tissues...\n")
print(f"{'Tissue':<15} {'T2/T1':<10} {'Standard MSE':<15} {'Conditional MSE':<18} {'Winner'}")
print("-" * 70)

with torch.no_grad():
    for name, t1, t2 in test_tissues_eval:
        ratio = t2 / t1
        signal = generate_signal_simple(t1, t2, params, width=width)
        
        # Standard GASP
        signal_2d = signal.reshape(width, 1, -1)
        std_output = run_gasp(signal_2d, standard_coeffs, method="affine")
        std_output_flat = np.abs(std_output.flatten())
        std_mse = np.mean((std_output_flat - target_profile) ** 2)
        
        # Conditional GASP
        signal_t = torch.from_numpy(signal).to(device)
        A, _ = model(signal_t, return_latent=True)
        Phi = design_matrix_torch(signal_t, model.method)
        cond_output = (Phi * A).sum(dim=-1).abs().cpu().numpy()
        cond_mse = np.mean((cond_output - target_profile) ** 2)
        
        # Store results
        results['tissue'].append(name)
        results['t1'].append(t1)
        results['t2'].append(t2)
        results['ratio'].append(ratio)
        results['standard_mse'].append(std_mse)
        results['conditional_mse'].append(cond_mse)
        results['standard_output'].append(std_output_flat)
        results['conditional_output'].append(cond_output)
        
        winner = "Conditional" if cond_mse < std_mse else "Standard"
        print(f"{name:<15} {ratio:<10.4f} {std_mse:<15.6f} {cond_mse:<18.6f} {winner}")

In [ ]:
# Summary statistics
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)

std_mean = np.mean(results['standard_mse'])
cond_mean = np.mean(results['conditional_mse'])
improvement = (std_mean - cond_mean) / std_mean * 100

print(f"Standard GASP mean MSE: {std_mean:.6f}")
print(f"Conditional GASP mean MSE: {cond_mean:.6f}")
print(f"Average improvement: {improvement:+.1f}%")

wins = sum(1 for s, c in zip(results['standard_mse'], results['conditional_mse']) if c < s)
print(f"Conditional GASP wins: {wins}/{len(results['tissue'])} tissues")

In [ ]:
# Plot MSE comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1 = axes[0]
x = np.arange(len(results['tissue']))
bar_width = 0.35

bars1 = ax1.bar(x - bar_width/2, results['standard_mse'], bar_width,
                label='Standard GASP', color='steelblue')
bars2 = ax1.bar(x + bar_width/2, results['conditional_mse'], bar_width,
                label='Conditional GASP', color='coral')

ax1.set_xlabel('Tissue Type')
ax1.set_ylabel('MSE')
ax1.set_title('GASP Performance Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels(results['tissue'], rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# MSE vs T2/T1 ratio
ax2 = axes[1]
ax2.scatter(results['ratio'], results['standard_mse'],
            s=100, label='Standard GASP', color='steelblue', alpha=0.7)
ax2.scatter(results['ratio'], results['conditional_mse'],
            s=100, label='Conditional GASP', color='coral', alpha=0.7)

for i, name in enumerate(results['tissue']):
    ax2.annotate(name, (results['ratio'][i], results['standard_mse'][i]),
                 textcoords="offset points", xytext=(5, 5), fontsize=8)

ax2.set_xlabel('T2/T1 Ratio')
ax2.set_ylabel('MSE')
ax2.set_title('MSE vs T2/T1 Ratio')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

## 5. Visualize Spectral Profiles

In [ ]:
# Plot spectral profiles for each tissue
n_tissues = len(results['tissue'])
n_cols = 3
n_rows = (n_tissues + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

freq_axis = np.linspace(-1, 1, width)

for i, tissue in enumerate(results['tissue']):
    ax = axes[i]
    
    ax.plot(freq_axis, target_profile, 'k--', label='Target', linewidth=2)
    ax.plot(freq_axis, results['standard_output'][i], 'b-', 
            label=f'Standard (MSE={results["standard_mse"][i]:.4f})', alpha=0.7)
    ax.plot(freq_axis, results['conditional_output'][i], 'r-',
            label=f'Conditional (MSE={results["conditional_mse"][i]:.4f})', alpha=0.7)
    
    ax.set_xlabel('Normalized Frequency')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{tissue} (T2/T1={results["ratio"][i]:.4f})')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide empty subplots
for i in range(n_tissues, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Analyze Latent Space

In [ ]:
# Generate embeddings across T2/T1 ratio range
ratios = np.logspace(-2, np.log10(0.5), 20)  # 0.01 to 0.5
T1_fixed = 1.0

embeddings = []
with torch.no_grad():
    for ratio in ratios:
        T2 = T1_fixed * ratio
        signal = generate_signal_simple(T1_fixed, T2, params, width=width)
        signal_t = torch.from_numpy(signal).to(device)
        _, z = model(signal_t, return_latent=True)
        embeddings.append(z.mean(dim=0).cpu().numpy())

embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Visualize latent space using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA plot colored by T2/T1 ratio
ax1 = axes[0]
scatter = ax1.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                      c=np.log10(ratios), cmap='viridis', s=100)
plt.colorbar(scatter, ax=ax1, label='log10(T2/T1)')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title('Latent Space (PCA) colored by T2/T1 ratio')
ax1.grid(True, alpha=0.3)

# Plot path through latent space
ax1.plot(embeddings_2d[:, 0], embeddings_2d[:, 1], 'k-', alpha=0.3)

# Embedding magnitude vs T2/T1 ratio
ax2 = axes[1]
embedding_norms = np.linalg.norm(embeddings, axis=1)
ax2.plot(ratios, embedding_norms, 'o-')
ax2.set_xlabel('T2/T1 Ratio')
ax2.set_ylabel('Embedding Norm')
ax2.set_title('Embedding Magnitude vs T2/T1 Ratio')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of latent dimensions vs T2/T1 ratio
plt.figure(figsize=(12, 6))
plt.imshow(embeddings.T, aspect='auto', cmap='RdBu_r',
           extent=[np.log10(ratios[0]), np.log10(ratios[-1]), 0, embeddings.shape[1]])
plt.colorbar(label='Activation')
plt.xlabel('log10(T2/T1 Ratio)')
plt.ylabel('Latent Dimension')
plt.title('Latent Space Activations vs T2/T1 Ratio')
plt.show()

## 7. Save Model (Optional)

In [ ]:
# Save the trained model
save_path = project_root / 'models' / 'conditional_gasp.pt'
save_path.parent.mkdir(exist_ok=True)

model.save(save_path)
print(f"Model saved to {save_path}")

In [ ]:
# Load and verify
loaded_model = ConditionalGASP.load(save_path, device=device)
print(f"Model loaded successfully")
print(f"  n_acquisitions: {loaded_model.n_acquisitions}")
print(f"  latent_dim: {loaded_model.latent_dim}")
print(f"  method: {loaded_model.method}")